# French election forecasts — FiftyPlusOne-style model

Replicates the method in *How the FiftyPlusOne 2026 election model works* for France (see `PLAN.md`):
polls model (three-pass average, house effects, lineup now-cast) + fundamentals (Stan softmax DLM),
stacked by out-of-sample precision, with an error meta-model fitted on the model's own backtest mistakes
and 40,000 simulations of both rounds.

This notebook is configuration only; the logic lives in the `forecast` package. Rebuild inputs with
`python -m forecast ingest`, backtests with `python -m forecast backtest` / `legi-backtest`.

In [ ]:
import warnings
from datetime import date
import polars as pl
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
pl.Config.set_tbl_rows(20)

from forecast.presidential.pipeline import run_forecast
from forecast.legislative.pipeline import run_legislative_forecast
from forecast.report import charts
from forecast.backtest.report import write_report

## Configuration

In [ ]:
AS_OF = date(2026, 9, 28)   # forecast date: polls published by then
N_SIMS = 40_000
# Lineup and alliance probabilities: data/reference/scenarios_pres_2027.yaml, scenarios_legi.yaml
# Model parameters (with provenance): config/model.yaml

## 2027 presidential election

In [ ]:
pres = run_forecast(2027, as_of=AS_OF, n_sims=N_SIMS, write=True)
print(pres.report())

In [ ]:
charts.presidential_probabilities(pres.candidates); plt.show()
charts.presidential_shares(pres.candidates); plt.show()

In [ ]:
pres.matchups.head(10)

In [ ]:
pres.lineups.head(10)   # the scenario mixture and whether each lineup has been polled

## Next legislative election (date assumed in `scenarios_legi.yaml`)

In [ ]:
legi = run_legislative_forecast(as_of=AS_OF, n_sims=N_SIMS, write=True)
print(legi.report())
charts.legislative_seats(legi.totals); plt.show()

In [ ]:
{name: df for name, df in legi.by_scenario.items()}

## Backtest (strict out-of-sample, 2002–2022)

In [ ]:
archive = pl.read_parquet('data/archive/pres_backtest.parquet')
print(write_report(archive))

In [ ]:
from IPython.display import Image, display
for f in ('error_by_horizon', 'weights_over_time', 'calibration'):
    display(Image(f'data/outputs/backtest_pres/{f}.png'))